In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision 
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image => scale(0,1) => normalize => (-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

C:\Users\Krishna gupta\AppData\Roaming\Python\Python313\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
trainloader = DataLoader(trainset, batch_size=64,shuffle=True)
testloader = DataLoader(testset, batch_size=64)


### Build the CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # first convolutional layer
            nn.Conv2d(3,32 , kernel_size=3,padding=1), #input ,output,...,...
            nn.ReLU(),
            nn.MaxPool2d(2,2),# kernel_size = 2,stride = 2

            # second convolutional layer
            nn.Conv2d(32,64 , kernel_size=3,padding=1), #input ,output,...,...
            nn.ReLU(),
            nn.MaxPool2d(2,2),# kernel_size = 2,stride = 2

            # third convolutional layer
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        ) 
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)

        return x

In [5]:
model =CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [9]:
epochs  = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # forward propagation
        loss = criterion(output,labels) #loss fnx
        loss.backward() # back propagation
        optimizer.step() #update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss=0.5011886253266993
epoch = 2/10 & loss=0.40763727223019464
epoch = 3/10 & loss=0.32371805879809057
epoch = 4/10 & loss=0.25103340143590325
epoch = 5/10 & loss=0.19548379891859297
epoch = 6/10 & loss=0.1480286619852266
epoch = 7/10 & loss=0.1291996246892149
epoch = 8/10 & loss=0.11284575788566219
epoch = 9/10 & loss=0.1000359996776942
epoch = 10/10 & loss=0.08453165803356524


### Evaluate our CNN

In [11]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels *100}")
        

accuracy = 75.07000000000001


In [32]:
model.eval()
running_val_loss = 0.0

with torch.no_grad():  # no gradients compute
        for images, labels in testloader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

epoch_val_loss = running_val_loss / len(testloader)  
    

print(f"Validation Loss = {epoch_val_loss}")

Validation Loss = 1.4749642610549927
